In [2]:
import os
os.listdir()

['trains.json',
 'stations.json',
 'Train_Throughput_AI_Project.ipynb',
 'railway_preprocessing.ipynb',
 'schedules.json',
 '.ipynb_checkpoints',
 'indian-railways-dataset.zip']

In [3]:
import os
print(os.getcwd())

/Users/derekdsouza/railway-ml


In [4]:
os.listdir()


['trains.json',
 'stations.json',
 'Train_Throughput_AI_Project.ipynb',
 'railway_preprocessing.ipynb',
 'schedules.json',
 '.ipynb_checkpoints',
 'indian-railways-dataset.zip']

In [5]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt

Matplotlib is building the font cache; this may take a moment.


In [6]:
import pandas as pd
import numpy as np
import json

In [7]:
with open('schedules.json') as f:
    schedules = json.load(f)

df = pd.DataFrame(schedules)

df.head()

,arrival,day,train_name,station_name,station_code,id,train_number,departure
0,None,1.0,Falaknuma Lingampalli MMTS,KACHEGUDA FALAKNUMA,FM,302214,47154,07:55:00
1,None,1.0,Thrissur Guruvayur Passenger,THRISUR,TCR,281458,56044,18:55:00
2,None,1.0,Porbandar Muzaffarpur Express,PORBANDAR,PBR,309335,19269,15:05:00
3,None,1.0,RAIPUR ITWARI PASS,RAIPUR JN,R,283774,58205,13:30:00
4,None,1.0,Gomoh-Asansol MEMU,GOMOH JN,GMO,319937,63542,07:20:00


In [8]:
df.head




<bound method NDFrame.head of          arrival  day                     train_name         station_name  \
0           None  1.0     Falaknuma Lingampalli MMTS  KACHEGUDA FALAKNUMA   
1           None  1.0   Thrissur Guruvayur Passenger              THRISUR   
2           None  1.0  Porbandar Muzaffarpur Express            PORBANDAR   
3           None  1.0             RAIPUR ITWARI PASS            RAIPUR JN   
4           None  1.0             Gomoh-Asansol MEMU             GOMOH JN   
...          ...  ...                            ...                  ...   
417075  12:39:00  2.0    HOWRAH - JAYNAGAR Passenger            RAJANAGAR   
417076  12:46:00  2.0    HOWRAH - JAYNAGAR Passenger      LALIT LAKSHMIPR   
417077  12:59:00  2.0    HOWRAH - JAYNAGAR Passenger             KHAJAULI   
417078  13:09:00  2.0    HOWRAH - JAYNAGAR Passenger              KORAHIA   
417079  14:20:00  2.0    HOWRAH - JAYNAGAR Passenger             JAYNAGAR   

       station_code      id train_number depa

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [9]:
df.head()

,arrival,day,train_name,station_name,station_code,id,train_number,departure
0,None,1.0,Falaknuma Lingampalli MMTS,KACHEGUDA FALAKNUMA,FM,302214,47154,07:55:00
1,None,1.0,Thrissur Guruvayur Passenger,THRISUR,TCR,281458,56044,18:55:00
2,None,1.0,Porbandar Muzaffarpur Express,PORBANDAR,PBR,309335,19269,15:05:00
3,None,1.0,RAIPUR ITWARI PASS,RAIPUR JN,R,283774,58205,13:30:00
4,None,1.0,Gomoh-Asansol MEMU,GOMOH JN,GMO,319937,63542,07:20:00


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 417080 entries, 0 to 417079
Data columns (total 8 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   arrival       417080 non-null  object 
 1   day           394519 non-null  float64
 2   train_name    417080 non-null  object 
 3   station_name  417080 non-null  object 
 4   station_code  417080 non-null  object 
 5   id            417080 non-null  int64  
 6   train_number  417080 non-null  object 
 7   departure     417080 non-null  object 
dtypes: float64(1), int64(1), object(6)
memory usage: 25.5+ MB


In [11]:
df['arrival'] = pd.to_datetime(df['arrival'], errors='coerce')
df['departure'] = pd.to_datetime(df['departure'], errors='coerce')

/var/folders/jq/7mz7h2f9643255w992tsk2140000gn/T/ipykernel_21020/4109724673.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['arrival'] = pd.to_datetime(df['arrival'], errors='coerce')
/var/folders/jq/7mz7h2f9643255w992tsk2140000gn/T/ipykernel_21020/4109724673.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['departure'] = pd.to_datetime(df['departure'], errors='coerce')


In [13]:
df = df.dropna(subset=['arrival', 'departure', 'day']).copy()

df.loc[:, 'day'] = df['day'].astype(int)

df.loc[:, 'delay'] = (df['arrival'] - df['departure']).dt.total_seconds() / 60

df.loc[:, 'hour'] = df['departure'].dt.hour
df.loc[:, 'minute'] = df['departure'].dt.minute

In [14]:
df[['delay', 'hour', 'day']].head()

,delay,hour,day
17,-1.0,6,1
27,-5.0,18,1
297,-10.0,13,1
561,-5.0,9,1
718,-20.0,11,1


In [15]:
throughput = df.groupby(['station_code', 'hour']).size().reset_index(name='train_count')

throughput.head()

,station_code,hour,train_count
0,AA,5,2
1,AA,6,1
2,AA,7,2
3,AA,8,1
4,AA,11,3


In [16]:
def classify_traffic(x):
    if x > 10:
        return 2   # High traffic
    elif x > 5:
        return 1   # Medium
    else:
        return 0   # Low

throughput['traffic_level'] = throughput['train_count'].apply(classify_traffic)


In [17]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
throughput['station_code'] = le.fit_transform(throughput['station_code'])

X = throughput[['station_code', 'hour']]
y = throughput['traffic_level']

In [18]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier()
model.fit(X_train, y_train)

RandomForestClassifier()

In [19]:
from sklearn.metrics import accuracy_score

pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred))

Accuracy: 0.6942647505752616


In [20]:
throughput['predicted'] = model.predict(X)

In [21]:
def control(level):
    if level == 2:
        return "Hold / Delay Train"
    elif level == 1:
        return "Adjust Speed"
    else:
        return "Allow Movement"

throughput['decision'] = throughput['predicted'].apply(control)

In [22]:
throughput.head()

,station_code,hour,train_count,traffic_level,predicted,decision
0,0,5,2,0,1,Adjust Speed
1,0,6,1,0,0,Allow Movement
2,0,7,2,0,0,Allow Movement
3,0,8,1,0,0,Allow Movement
4,0,11,3,0,0,Allow Movement


In [23]:
throughput = df.groupby(['station_code', 'hour']).size().reset_index(name='train_count')
throughput.head()

,station_code,hour,train_count
0,AA,5,2
1,AA,6,1
2,AA,7,2
3,AA,8,1
4,AA,11,3


In [24]:
def classify_traffic(x):
    if x >= 3:
        return 2   # High traffic
    elif x == 2:
        return 1   # Medium traffic
    else:
        return 0   # Low traffic

throughput['traffic_level'] = throughput['train_count'].apply(classify_traffic)

In [25]:
throughput.head()

,station_code,hour,train_count,traffic_level
0,AA,5,2,1
1,AA,6,1,0
2,AA,7,2,1
3,AA,8,1,0
4,AA,11,3,2


In [26]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
throughput['station_code'] = le.fit_transform(throughput['station_code'])

In [27]:
X = throughput[['station_code', 'hour']]
y = throughput['traffic_level']

In [28]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier()
model.fit(X_train, y_train)

RandomForestClassifier()

In [29]:
from sklearn.metrics import accuracy_score

pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred))

Accuracy: 0.3799765553770677


In [30]:
print("Accuracy:", accuracy_score(y_test, pred))

Accuracy: 0.3799765553770677


In [31]:
model = RandomForestClassifier(n_estimators=200, max_depth=10)
model.fit(X_train, y_train)

pred = model.predict(X_test)

from sklearn.metrics import accuracy_score
print("New Accuracy:", accuracy_score(y_test, pred))

New Accuracy: 0.4796162028394043


In [32]:
import pickle

with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

throughput.to_csv("throughput.csv", index=False)